# Đánh giá Mô hình nhận dạng Tiếng Nhật (Japanese ASR Evaluation)

Notebook này giúp bạn dễ dàng chạy và so sánh chất lượng nhận dạng Tiếng Nhật giữa Whisper mặc định và **Kotoba-Whisper** trực tiếp trên Google Colab với GPU miễn phí. Kết quả cuối cùng sẽ được trực quan hóa bằng các biểu đồ đánh giá chuyên sâu.

## 1. Cài đặt thư viện
Chạy ô dưới đây để cài đặt `faster-whisper`, thư viện tính toán Levenshtein và `matplotlib` để vẽ biểu đồ.

In [ ]:
!pip install faster-whisper Levenshtein soundfile matplotlib

## 2. Chuẩn bị Hàm tính CER (Character Error Rate)

In [ ]:
import time
import re
from faster_whisper import WhisperModel
import Levenshtein
import matplotlib.pyplot as plt
import numpy as np

def normalize_text_ja(text):
    """Normalize Japanese text (lowercase, remove punctuation and spaces)."""
    text = text.lower()
    text = re.sub(r'[^\w]', '', text)
    return text

def calculate_cer(ref, hyp):
    """Tính tỉ lệ lỗi ký tự (CER)."""
    ref_norm = normalize_text_ja(ref)
    hyp_norm = normalize_text_ja(hyp)
    
    if not ref_norm:
        return 1.0 if hyp_norm else 0.0
        
    dist = Levenshtein.distance(ref_norm, hyp_norm)
    return dist / len(ref_norm)


## 3. Tải Audio Mẫu (Sample Audio)
Bạn có thể tải lên một file audio Tiếng Nhật bằng nút Upload bên thanh công cụ trái, hoặc chạy ô dưới đây để thử tạo một đoạn audio mẫu giả định.

*(Lưu ý: Thay `audio_path` và `ground_truth` thành dữ liệu thực tế của bạn)*

In [ ]:
# Thay biến này bằng đường dẫn file audio tải lên của bạn
audio_path = "sample_ja.wav"

# Nếu bạn không có sẵn file, ô này sẽ tải một file âm thanh tiếng Anh nhỏ để bạn code không bị lỗi (Hãy upload file tiếng Nhật thực tế để kiểm tra model Nhật)
import os
if not os.path.exists(audio_path):
    !wget -q -O sample_ja.wav "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"

# Bạn nhập văn bản gốc (Ground Truth) tương ứng với audio vào đây để phần mềm chấm điểm CER
ground_truth = "こんにちは、今日の天気は晴れです。" # <--- Đổi thành transcript đúng của file audio


## 4. Tải Mô hình lên GPU

In [ ]:
print("Loading default Whisper (small)...")
model_default = WhisperModel("small", device="cuda", compute_type="float16")

print("\nLoading Kotoba-Whisper (v2.0 faster)...")
model_kotoba = WhisperModel("kotoba-tech/kotoba-whisper-v2.0-faster", device="cuda", compute_type="float16")


## 5. Tiến hành Đánh giá

In [ ]:
evaluation_results = []

def evaluate_model(model, name, audio_path, ground_truth):
    print(f"\n{'='*50}")
    print(f"--- Đang chạy nhận dạng với {name} ---")
    start_time = time.time()
    
    segments, info = model.transcribe(audio_path, language="ja", beam_size=5)
    hyp_text = " ".join([seg.text for seg in segments])
    
    elapsed_time = time.time() - start_time
    cer = calculate_cer(ground_truth, hyp_text)
    
    print(f"Kết quả nhận dạng : {hyp_text}")
    print(f"Văn bản chuẩn (GT): {ground_truth}")
    print(f"Thời gian xử lý   : {elapsed_time:.2f}s")
    print(f"CER (Lỗi ký tự)   : {cer * 100:.2f}%")
    print(f"{'='*50}\n")
    
    return {
        "Model": name,
        "CER": cer * 100,
        "Time": elapsed_time
    }

res_default = evaluate_model(model_default, "Whisper (small)", audio_path, ground_truth)
res_kotoba = evaluate_model(model_kotoba, "Kotoba Whisper (v2.0)", audio_path, ground_truth)

evaluation_results.extend([res_default, res_kotoba])


## 6. Trực quan hóa Biểu đồ (Visualization)
Biểu đồ so sánh tỉ lệ lỗi (CER) và tốc độ xử lý (Latency) giữa hai mô hình.

In [ ]:
models = [r["Model"] for r in evaluation_results]
cer_scores = [r["CER"] for r in evaluation_results]
times = [r["Time"] for r in evaluation_results]

x = np.arange(len(models))
width = 0.35

fig, ax1 = plt.subplots(figsize=(10, 6))

# Trục Y 1: Đo lường CER
color1 = 'tab:red'
ax1.set_xlabel('Mô hình (Model)', fontsize=12)
ax1.set_ylabel('CER (%) - Càng thấp càng tốt', color=color1, fontsize=12)
bars1 = ax1.bar(x - width/2, cer_scores, width, label='CER (%)', color=color1, alpha=0.8)
ax1.tick_params(axis='y', labelcolor=color1)

# Gắn nhãn giá trị trên cột CER
for bar in bars1:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + (yval * 0.05 + 0.1), f'{yval:.1f}%', ha='center', va='bottom', color=color1, fontweight='bold')

# Trục Y 2: Đo lường Thời gian (Latency)
ax2 = ax1.twinx()  
color2 = 'tab:blue'
ax2.set_ylabel('Thời gian xử lý (s) - Càng thấp càng tốt', color=color2, fontsize=12)
bars2 = ax2.bar(x + width/2, times, width, label='Thời gian (s)', color=color2, alpha=0.8)
ax2.tick_params(axis='y', labelcolor=color2)

# Gắn nhãn giá trị trên cột Thời gian
for bar in bars2:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, yval + (yval * 0.05 + 0.1), f'{yval:.2f}s', ha='center', va='bottom', color=color2, fontweight='bold')

plt.title('So sánh Độ chính xác (CER) và Tốc độ xử lý giữa các mô hình', fontsize=14, fontweight='bold', pad=20)
plt.xticks(x, models)

fig.tight_layout()
plt.show()
